In [1]:
import numpy as np
import cirq

def run_simple_xeb(n_qubits=4, cycles=10):
    # 1. Setup Qubits and Circuit
    qubits = cirq.GridQubit.square(2)  # 2x2 grid for 4 qubits
    circuit = cirq.Circuit()
    
    # 2. Build a Random Circuit (Simplified)
    # In a real XEB, we use a specific sequence of single and two-qubit gates
    for _ in range(cycles):
        # Add random single-qubit gates (Fix: use list instead of generator)
        circuit.append([cirq.PhasedXPowGate(
            exponent=np.random.random(), 
            phase_exponent=np.random.random()
        ).on(q) for q in qubits])
        
        # Add entangling gates (CZ) in a grid pattern
        circuit.append(cirq.CZ(qubits[0], qubits[1]))
        circuit.append(cirq.CZ(qubits[2], qubits[3]))

    # 3. Get "Ideal" Probabilities from Simulation
    simulator = cirq.Simulator()
    result = simulator.simulate(circuit)
    # The full state vector probabilities
    probs_ideal = np.abs(result.state_vector())**2

    # 4. "Sample" from the Quantum Device (Simulated Noise-Free here)
    # In practice, this step happens on the actual hardware
    # Make a copy of circuit before adding measurement
    circuit_with_measure = circuit.copy()
    circuit_with_measure.append(cirq.measure(*qubits, key='m'))
    samples = simulator.sample(circuit_with_measure, repetitions=1000)
    
    # 5. Calculate XEB Fidelity
    # Formula: F = 2^n * <P_ideal(observed_bitstring)> - 1
    observed_bitstrings = samples['m'].values
    # Convert bitstring arrays to integer indices
    # Fix: properly handle the bitstring conversion
    indices = []
    for bitstring in observed_bitstrings:
        # bitstring is a numpy array or list, convert each row properly
        if hasattr(bitstring, '__len__'):
            idx = int(''.join(map(str, bitstring[::-1])), 2)
        else:
            # Single value case
            idx = int(bitstring)
        indices.append(idx)
    
    # Mean of ideal probabilities at the points we actually observed
    mean_p_ideal = np.mean([probs_ideal[idx] for idx in indices])
    
    n = n_qubits
    fidelity = (2**n) * mean_p_ideal - 1
    
    return fidelity

# Run the experiment
f_score = run_simple_xeb()
print(f"Estimated XEB Fidelity: {f_score:.4f}")

Estimated XEB Fidelity: 2.1034
